# Model Diagnostic Suite

Quick health checks and diagnostics: model health, computation budget,
prediction quality, attention health, and residual stream stability.

## Why This Matters

Model diagnostic suite provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_diagnostic_suite import (
    model_health_check, computation_budget_profile,
    prediction_quality_summary, attention_health_summary,
    residual_stream_health,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Model Health Check

Check for NaN, Inf, and extreme values.

In [ ]:
result = model_health_check(model, tokens)
status = 'HEALTHY' if result['is_healthy'] else 'ISSUES FOUND'
print(f"Status: {status}")
print(f"Max logit magnitude: {result['max_logit_magnitude']:.4f}")
print(f"Issues: {result['n_issues']}\n")
for s in result['layer_stats']:
    print(f"  Layer {s['layer']}: max_val={s['max_value']:.4f}, "
          f"mean_norm={s['mean_norm']:.4f}, nan={s['has_nan']}")
for issue in result['issues']:
    print(f"  WARNING: {issue}")

## Computation Budget

Where is computation spent?

In [ ]:
result = computation_budget_profile(model, tokens)
print(f"Total computation: {result['total_computation']:.4f}")
print(f"Top component: {result['top_component']}\n")
for c in result['per_component']:
    bar = '█' * int(c['fraction'] * 40)
    print(f"  {c['name']:10s} ({c['type']:4s}): {c['magnitude']:.4f} "
          f"({c['fraction']:.1%}) {bar}")

## Prediction Quality

How confident is the model at each position?

In [ ]:
result = prediction_quality_summary(model, tokens)
print(f"Mean entropy: {result['mean_entropy']:.4f}")
print(f"Mean confidence: {result['mean_confidence']:.4f}")
print(f"Most confident: pos {result['most_confident_position']}")
print(f"Least confident: pos {result['least_confident_position']}\n")
for p in result['per_position']:
    bar = '█' * int(p['confidence'] * 30)
    print(f"  Pos {p['position']}: entropy={p['entropy']:.4f}, "
          f"top={p['top_token']}, conf={p['confidence']:.4f}, "
          f"margin={p['margin']:.4f} {bar}")

## Attention Health

Are attention patterns healthy or degenerate?

In [ ]:
result = attention_health_summary(model, tokens)
print(f"Healthy: {result['n_healthy']}, Degenerate: {result['n_degenerate']}, BOS sinks: {result['n_bos_sinks']}\n")
for h in result['per_head']:
    flags = []
    if h['is_degenerate']: flags.append('DEGENERATE')
    if h['is_bos_sink']: flags.append('BOS_SINK')
    tag = ', '.join(flags) if flags else 'ok'
    print(f"  L{h['layer']}H{h['head']}: entropy={h['mean_entropy']:.4f}, "
          f"max_attn={h['mean_max_attention']:.4f}, bos={h['bos_weight']:.4f} [{tag}]")

## Residual Stream Health

Is the residual stream stable or exploding?

In [ ]:
result = residual_stream_health(model, tokens)
status = 'STABLE' if result['is_stable'] else 'UNSTABLE'
print(f"Status: {status}")
print(f"Max growth rate: {result['max_growth_rate']:.4f}")
print(f"Final norm: {result['final_norm']:.4f}\n")
for p in result['per_layer']:
    flag = 'EXPLODING' if p['is_exploding'] else 'ok'
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"std={p['std_norm']:.4f}, growth={p['growth_rate']:.4f} [{flag}]")